# The main Contracting client

Contracting ships with a client that abstracts most of the low-level runtime mechanics so that working with contracts feels like regular Python.


In [ ]:
from contracting.local import ContractingClient

client = ContractingClient(signer="stu")
client.flush()


## Viewing contracts

The client preloads a set of 'genesis' contracts that are required for the system to operate. These genesis contracts bypass some of the system protections to allow a channel between the low level protocol and the higher level programming interface to occur without exposing everything to the developer. This allows us to create powerful system level contracts while maintaining tight security on user level contracts.

One of these genesis contracts is the contract submission contract. Ironically, we have a smart contract to submit smart contracts. This is for purposes of conflict resolution, which is a later topic. For now, you can see the contracts by calling the function below.

In [ ]:
client.get_contracts()

## Interacting with existing smart contracts

Contracts can be fetched into their own objects with the `get_contract_proxy` method.

In [ ]:
client.get_contract_proxy('submission')

You can see all available functions you can call on this smart contract by accessing the `self.functions` property. This returns a list of tuples made up of function names and the list of keyword arguments they require.

In [ ]:
submission = client.get_contract_proxy('submission')
submission.functions

## Submitting new smart contracts

Let's use the client to submit a new smart contract. Contract names must be lowercase and start with `con_`.


In [ ]:
code = """
@export
def hello_there():
    return "howdy"
"""

name = "con_howdy_time"
client.submit(code, name=name)


Now we can check to make sure the contract made it into its own namespace. Every contract stores human-facing source in `__source__` and canonical Xian VM IR in `__xian_ir_v1__`.


In [ ]:
howdy_time = client.get_contract_proxy(name)

In [ ]:
howdy_time.keys()

In [ ]:
print(howdy_time.__source__)


It worked. `__source__` gives you the normalized source that was submitted. The runtime also stores canonical executable IR in `__xian_ir_v1__`.


In [ ]:
code = """
@export
def secret():
    return abcd()


def abcd():
    return 5
"""

name = "con_secret"
client.submit(code, name=name)


In [ ]:
secret = client.get_contract_proxy("con_secret")


In [ ]:
secret.functions

In [ ]:
print(client.raw_driver.get_contract_ir("con_secret")[:500])


In [ ]:
secret.secret()

Notice how methods without `@export` are private in the contract interface even though the original submitted source stays clean in `__source__`.


## Direct submission

Instead of talking to the `submission` contract directly each time, you can submit source strings through `client.submit(...)`. This is the most portable notebook-friendly path because it does not rely on source introspection from live notebook cells.


In [ ]:
closure_example = """
@export
def hello():
    return "How are you?"


def shh():
    return 10
"""


In [ ]:
client.submit(closure_example, name="con_closure_example")


In [ ]:
closure = client.get_contract_proxy("con_closure_example")


In [ ]:
print(client.raw_driver.get_contract_ir("con_closure_example")[:500])

Now that you have an idea on how to use the client, let's dive into the mechanics behind smart contracts in the next notebook.